In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# 1. VERİYİ OKUMA 
df = pd.read_csv('emails.csv')

X = df['text'].values
y = df['spam'].values

# 2. TRAIN - TEST SPLIT (Veriyi %75 Eğitim, %25 Test olarak ayırma)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 3. TEXT PREPROCESSING (Metni Sayılara Çevirme)
# CountVectorizer: Metinleri alır, içindeki kelimeleri sayar ve bir sayısal matris (tablo) oluşturur.
cv = CountVectorizer()

# Eğitim verisini hem öğretip (fit) hem sayılara çeviriyoruz (transform)
X_train_vectorized = cv.fit_transform(X_train)
# Test verisini ise ÖĞRETMEDEN (sadece transform) sayılara çeviriyoruz 
# (Çünkü model bilmediği kelimelerle test edilmeli)
X_test_vectorized = cv.transform(X_test)

# 4. MODEL KURULUMU VE EĞİTİM (MultinomialNB)
# Metin (kelime sayımı) projeleri için en iyisi Multinomial Naive Bayes'tir
classifier = MultinomialNB()
classifier.fit(X_train_vectorized, y_train)

# 5. PREDICTION 
y_pred = classifier.predict(X_test_vectorized)

# 6. SONUÇLARI İNCELEME
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)





Confusion Matrix:
 [[1055    8]
 [  11  358]]


# Premium Mini Project: Email Spam Detection System

## AŞAMA 1 — Problem Understanding
**Spam tespiti neden classification problemidir?**
E-posta sistemlerinde spam tespiti, makine öğrenmesi dünyasının en klasik ve en saf "Classification" (Sınıflandırma) problemlerinden biridir. Bunun temel nedeni, algoritmanın ulaşmaya çalıştığı nihai hedefin sürekli artan veya azalan sayısal bir değer (maaş, ev fiyatı veya şirket kârı gibi) olmamasıdır. Karşımızdaki problem matematiksel bir tahmin (Regresyon) değil, kategorik bir karar verme sürecidir. Algoritmadan istenen şey, gelen bir e-postanın içeriğini analiz edip onu önceden tanımlanmış iki farklı kutudan (kategoriden) birine atmasıdır: Ya "0 - Normal (Ham) Mail" ya da "1 - İstenmeyen (Spam) Mail". Bu karar mekanizması, gri alanlara yer bırakmayan, tamamen siyah veya beyaz olan ikili bir sınıflandırma (binary classification) işlemidir. Sistem, metindeki kelimelerin olasılıklarını hesaplar ve günün sonunda e-postaya keskin bir etiket yapıştırır. Regresyon modelleri "Ne kadar?" sorusuna cevap verirken, sınıflandırma modelleri "Bu nedir?" sorusuna cevap verir. Spam tespiti de tam olarak e-postanın kimliğini ve kategorisini sorguladığı için yapısal olarak kesin bir sınıflandırma problemidir.

## AŞAMA 2 — Feature Thinking
Bir e-posta verisetinde spamleri yakalamak için sayısal ve mantıksal olarak çıkartılabilecek en güçlü özellikler (features) şunlar olabilir:

1. **`contains_link`**: Mail içinde dışarıya yönlendiren tıklanabilir bir URL bağlantısı var mı?
2. **`message_length`**: E-postanın toplam karakter veya kelime sayısı (Spamler genellikle ya çok kısa ya çok uzun olur).
3. **`contains_discount`**: Metin içinde "indirim", "fırsat", "%50" gibi promosyon tetikleyici kelimeler geçiyor mu?
4. **`contains_money_word`**: "Para", "kazanç", "kredi", "dolar" gibi finansal tuzak kelimeleri içeriyor mu?
5. **`sender_reputation`**: Gönderen kişinin e-posta adresinin veya domain'inin geçmiş güvenilirlik skoru.
6. **`uppercase_ratio`**: Konu başlığında veya metin içinde kullanılan BÜYÜK HARF oranı (Spam maillerde bağırma hissi vermek için çok sık görülür).
7. **`num_attachments`**: E-postaya eklenmiş olan dosya (PDF, exe, zip) sayısı (Virüslü eklentiler için şüphelidir).
8. **`has_suspicious_subject`**: Konu başlığının "Acil", "Önemli", "Tebrikler kazandınız" gibi sahte aciliyet bildiren kalıplar içerip içermediği.
9. **`is_html`**: Mailin sadece düz metin (plain text) mi yoksa karmaşık HTML ve görsel öğeler mi içerdiği.
10. **`recipient_count`**: Mailin aynı anda kaç kişiye gönderildiği (Özellikle CC veya BCC kısmındaki kalabalık liste spam işaretidir).

## AŞAMA 4 — Naive Bayes Comparison
**GaussianNB, MultinomialNB ve BernoulliNB arasındaki fark nedir?**
Naive Bayes algoritması, olasılık hesaplamalarını yaparken verinin yapısına (dağılımına) göre farklı matematiksel varsayımlarda bulunur. Bu nedenle problemi çözerken elimizdeki veri tipine en uygun Naive Bayes türünü seçmek, modelin başarısı için en kritik adımdır. Karşımıza çıkan üç temel varyasyon tamamen verinin sayısal karakteristiğine göre ayrışır:

* **GaussianNB (Gauss Naive Bayes):** Bu varyasyon, elimizdeki özelliklerin (features) sürekli sayılardan (continuous data) oluştuğunu ve bu sayıların "Normal Dağılım" (Çan eğrisi) gösterdiğini varsayar. Örneğin; e-postanın kelime uzunluğu, dosya boyutları veya müşterinin maaşı gibi virgüllü veya çok geniş aralıklara sahip sonsuz sürekli verilerle uğraşıyorsak bu modeli kullanırız.
* **MultinomialNB (Multinomial Naive Bayes):** Metin madenciliği (Text Mining) ve Doğal Dil İşleme (NLP) projelerinin tartışmasız kralıdır. Verilerin ayrıksı sayımlardan (discrete counts) oluştuğu durumlarda kullanılır. Bir e-postanın içindeki "bedava" kelimesinin 5 kez, "fırsat" kelimesinin 2 kez geçmesi gibi kelime frekans (sıklık) temelli hesaplamalar yapar. Spam tespiti ve duygu analizi (sentiment analysis) projelerinde matrisler üzerinde en başarılı sonucu verir.
* **BernoulliNB (Bernoulli Naive Bayes):** Tıpkı Multinomial gibi metin sınıflandırmada kullanılır ancak dünyayı sadece "0 ve 1" olarak görür. Verilerin yalnızca ikili (binary - boolean) değerler aldığı durumlar için tasarlanmıştır. "Bu kelime mailde kaç kere geçti?" sorusuyla ilgilenmez; sadece "Bu kelime mailde var mı (1) yok mu (0)?" sorusuna odaklanır.

## Executive Insight
**CEO soruyor: Yanlış spam tespiti mi daha tehlikeli, spam’i kaçırmak mı?**
Kesinlikle **yanlış spam tespiti (False Positive - Type 1 Error) çok daha tehlikelidir.** Bir spam maili kaçırıp (False Negative) müşterinin gelen kutusuna düşürürsek, müşteri bunu sadece silip geçer; bu ufak bir rahatsızlıktır. Ancak çok önemli bir iş teklifini, uçak biletini veya bir şifre sıfırlama e-postasını yanlışlıkla "spam" klasörüne atarak gizlersek, bu durum müşterinin şirketimizle olan tüm güven bağını koparır ve geri dönüşü olmayan krizlere yol açar. Spam filtrelerinde altın kural: *"Normal bir maili spam yapmaktansa, üç tane spam maili kaçırmak her zaman daha yeğdir."*

## Mentor Soruları
* **Naive Bayes neden hızlıdır?** Karmaşık mesafe hesaplamaları (KNN gibi) veya günlerce süren gradyan optimizasyonları (Derin Öğrenme gibi) yapmaz; sadece basit olasılık matematiği (geçmişi sayarak çarpma ve bölme) kullanır. Eğitimi milisaniyeler sürer.
* **“Naive” neden deniyor?** Naive (Saf/Cahil) denmesinin sebebi, e-postadaki kelimeleri birbirinden tamamen "bağımsız" varsaymasıdır. Algoritma "Kredi" ve "Kartı" kelimelerinin yan yana gelince bir anlam ifade ettiğini umursamaz (safça davranır), her bir kelimeye körü körüne ayrı ayrı olasılık hesaplar.
* **Probability mantığı neden önemli?** Çünkü yapay zeka, bir mailin spam olup olmadığını kesin kurallarla (if-else) değil, sadece geçmiş istatistiklerle bilebilir. Bayes Teoremi sayesinde algoritma, *"Geçmişte içinde 'ücretsiz' geçen maillerin %95'i spam çıkmıştı, o halde yeni gelen bu mailin de spam olma ihtimali çok yüksektir"* diyerek koşullu olasılığa dayanan muazzam bir mantık yürütür.